In [1]:
# # The maths behind XGBoost: numerical checks
# We check the derivation of the Note with numbers: the second-order Taylor approximation, the optimal leaf output
# w* = -G / (H + lambda), the minimum objective, the gain formula, and how the XGBoost library scales its gain.

In [2]:
import numpy as np
import xgboost as xgb
sigmoid = lambda z: 1 / (1 + np.exp(-z))

## 1. Squared error: the second-order Taylor approximation is exact

In [3]:
# one leaf of the regression Note: two students with residuals -2.875 and -1.375 from the mean 7.375
y = np.array([4.5, 6.0])
y_prev = np.full(2, 7.375)
loss = lambda w: (0.5 * (y - (y_prev + w)) ** 2).sum()     # L = 1/2 (y - yhat)^2
g = y_prev - y                                             # gradient: yhat - y
h = np.ones(2)                                             # Hessian: 1
taylor = lambda w: loss(0) + g.sum() * w + 0.5 * h.sum() * w ** 2
ws = np.linspace(-4, 2, 13)
print("largest gap between exact and Taylor:", max(abs(loss(w) - taylor(w)) for w in ws))

largest gap between exact and Taylor: 0.0


## 2. w* = -G / (H + lambda), checked by brute force on a fine grid

In [4]:
grid = np.linspace(-5, 5, 200001)
for lam in (0, 1):
    G, H = g.sum(), h.sum()
    w_star = -G / (H + lam)
    objective = [loss(w) + 0.5 * lam * w ** 2 for w in grid]
    print(f"lambda={lam}: formula w*={w_star:.4f}, grid minimum at {grid[np.argmin(objective)]:.4f}")

lambda=0: formula w*=-2.1250, grid minimum at -2.1250


lambda=1: formula w*=-1.4167, grid minimum at -1.4166


## 3. Log loss: the approximation is close, not exact

In [5]:
# left leaf of the classification Note: classes 0, 1, 0, all at log-odds ln 1.5
yc = np.array([0, 1, 0])
z0 = np.log(1.5)
p = sigmoid(z0)
logloss = lambda w: -(yc * np.log(sigmoid(z0 + w)) + (1 - yc) * np.log(1 - sigmoid(z0 + w))).sum()
G, H = (p - yc).sum(), len(yc) * p * (1 - p)             # g = p - y, h = p(1 - p) for each of the 3 rows
approx = lambda w: logloss(0) + G * w + 0.5 * H * w ** 2
for w in (-2, -1.11, 0, 1):
    print(f"w={w:+.2f}: exact {logloss(w):.4f}, second-order {approx(w):.4f}")
w_star = -G / H
w_grid = grid[np.argmin([logloss(w) for w in grid])]
print(f"w* from the formula {w_star:.4f}; true minimum of the log loss {w_grid:.4f}")

w=-2.00: exact 2.1490, second-order 2.1834
w=-1.11: exact 1.9096, second-order 1.8990
w=+0.00: exact 2.3434, second-order 2.3434
w=+1.00: exact 3.4689, second-order 3.5034


w* from the formula -1.1111; true minimum of the log loss -1.0986


## 4. Minimum objective and gain on the regression data

In [6]:
x = np.array([6.7, 9.0, 7.5, 5.0])
yr = np.array([4.5, 11.0, 6.0, 8.0])
gr = np.full(4, yr.mean()) - yr                            # gradients
hr = np.ones(4)                                            # Hessians

def best_obj(mask, lam=0.0, gamma=0.0):
    # -1/2 G^2 / (H + lambda) + gamma * T, for one leaf (T = 1)
    G, H = gr[mask].sum(), hr[mask].sum()
    return -0.5 * G ** 2 / (H + lam) + gamma

def gain(mask, parent, lam=0.0, gamma=0.0):
    # 1/2 [G_L^2/(H_L+lam) + G_R^2/(H_R+lam) - G_P^2/(H_P+lam)] - gamma
    sim = lambda m: gr[m].sum() ** 2 / (hr[m].sum() + lam)
    return 0.5 * (sim(mask & parent) + sim(~mask & parent) - sim(parent)) - gamma

everyone = np.ones(4, bool)
left = x < 8.25
print("objective before the split:", best_obj(everyone))
print("objective after the split :", best_obj(left) + best_obj(~left))
print("paper gain (with 1/2)     :", gain(left, everyone))
print("similarity gain (no 1/2)  :", 2 * gain(left, everyone))
print("lower split, paper gain   :", gain(x < 5.85, left))

objective before the split: 0.0
objective after the split : -8.760416666666666
paper gain (with 1/2)     : 8.760416666666666
similarity gain (no 1/2)  : 17.520833333333332
lower split, paper gain   : 2.5208333333333335


## 5. What the XGBoost library prints, and how it compares gamma
The dumped gain is the similarity gain without the 1/2. Gamma is compared with that number:
the lower split (gain 5.04) survives gamma = 5.0 and disappears at gamma = 5.1.

In [7]:
X = x.reshape(-1, 1)
for gamma in (0, 5.0, 5.1, 10, 18):
    m = xgb.XGBRegressor(n_estimators=1, max_depth=2, learning_rate=1.0, reg_lambda=0, gamma=gamma,
                         min_child_weight=0, base_score=yr.mean(), tree_method="exact", device="cpu").fit(X, yr)
    print(f"gamma={gamma}:")
    print(m.get_booster().get_dump(with_stats=True)[0])

gamma=0:
0:[f0<8.25] yes=1,no=2,missing=1,gain=17.520834,cover=4
	1:[f0<5.8499999] yes=3,no=4,missing=3,gain=5.04166651,cover=3
		3:leaf=0.625,cover=1
		4:leaf=-2.125,cover=2
	2:leaf=3.625,cover=1

gamma=5.0:
0:[f0<8.25] yes=1,no=2,missing=1,gain=17.520834,cover=4
	1:[f0<5.8499999] yes=3,no=4,missing=3,gain=5.04166651,cover=3
		3:leaf=0.625,cover=1
		4:leaf=-2.125,cover=2
	2:leaf=3.625,cover=1

gamma=5.1:
0:[f0<8.25] yes=1,no=2,missing=1,gain=17.520834,cover=4
	1:leaf=-1.20833337,cover=3
	2:leaf=3.625,cover=1

gamma=10:
0:[f0<8.25] yes=1,no=2,missing=1,gain=17.520834,cover=4
	1:leaf=-1.20833337,cover=3
	2:leaf=3.625,cover=1

gamma=18:
0:leaf=-0,cover=4



## 6. Does the Newton approximation of the leaf build up an error over many trees?
Same tree-growing rule for both runs; only the leaf values differ: XGBoost's Newton step $-G/H$ against the exact minimum of each leaf's log loss.

In [8]:
import numpy as np
from scipy.optimize import brentq
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
sig = lambda z: 1 / (1 + np.exp(-z))
logloss = lambda y, z: np.mean(np.log1p(np.exp(-z)) + (1 - y) * z)
X, y = make_classification(n_samples=2000, n_features=10, n_informative=5, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=0)
def boost(exact, M=100, eta=0.3):
    z0 = np.log(ytr.mean() / (1 - ytr.mean()))
    ztr, zte = np.full(len(ytr), z0), np.full(len(yte), z0)
    out = []
    for m in range(M):
        p = sig(ztr); g, h = p - ytr, p * (1 - p)
        # same tree structure rule for both: weighted least squares on -g/h with weights h (XGBoost's split score)
        t = DecisionTreeRegressor(max_depth=3, random_state=0).fit(Xtr, -g / h, sample_weight=h)
        leaf_tr, leaf_te = t.apply(Xtr), t.apply(Xte)
        w = {}
        for j in np.unique(leaf_tr):
            i = leaf_tr == j
            if exact and 0 < ytr[i].sum() < i.sum():
                # exact minimiser of the leaf's log loss: sum sigmoid(z_i + w) = sum y_i
                w[j] = brentq(lambda v: sig(ztr[i] + v).sum() - ytr[i].sum(), -50, 50)
            else:
                w[j] = -g[i].sum() / h[i].sum()      # Newton step (XGBoost, lambda = 0)
        ztr = ztr + eta * np.array([w[j] for j in leaf_tr])
        zte = zte + eta * np.array([w[j] for j in leaf_te])
        if m + 1 in (1, 10, 50, 100):
            out.append((m + 1, logloss(ytr, ztr), logloss(yte, zte)))
    return out
for name, ex in (("Newton (XGBoost)", False), ("exact leaf minimum", True)):
    for m, a, b in boost(ex):
        print(f"{name:20s} trees={m:3d}  train log loss {a:.4f}  test log loss {b:.4f}")


Newton (XGBoost)     trees=  1  train log loss 0.5271  test log loss 0.5451
Newton (XGBoost)     trees= 10  train log loss 0.1988  test log loss 0.2796
Newton (XGBoost)     trees= 50  train log loss 0.0574  test log loss 0.2614
Newton (XGBoost)     trees=100  train log loss 0.0158  test log loss 0.2818


exact leaf minimum   trees=  1  train log loss 0.4722  test log loss 0.4991
exact leaf minimum   trees= 10  train log loss 0.1699  test log loss 0.2677
exact leaf minimum   trees= 50  train log loss 0.0402  test log loss 0.2636
exact leaf minimum   trees=100  train log loss 0.0066  test log loss 0.3121
